In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

from src.utils.pipeline import load_all_snapshots
from src.features.plate_discipline import discipline_profile

df = load_all_snapshots()
profile = discipline_profile(df)

for k, v in profile.items():
    print(f"{k:20s} {v:.3f}" if isinstance(v, float) else f"{k:20s} {v:,}")

pitches              710,632
zone_pct             0.495
swing_pct            0.476
chase_pct            0.282
zone_swing_pct       0.674
contact_pct          0.768
zone_contact_pct     0.847
whiff_pct            0.232
n_out_of_zone        359,070
n_in_zone            351,562
n_swings             338,364
n_zone_swings        237,100


In [2]:
import pandas as pd

MIN_PITCHES = 500

rows = []
for batter_id, g in df.groupby("batter"):
    if len(g) < MIN_PITCHES:
        continue
    p = discipline_profile(g)
    p["batter"] = batter_id
    p["name"] = g["player_name"].iloc[0] if "player_name" in g.columns else None
    rows.append(p)

prof = pd.DataFrame(rows).set_index("batter")
print(f"{len(prof)} batters with >= {MIN_PITCHES} pitches")
print()
print(prof[["chase_pct", "zone_swing_pct", "contact_pct", "zone_contact_pct"]].describe().round(3))

425 batters with >= 500 pitches

       chase_pct  zone_swing_pct  contact_pct  zone_contact_pct
count    425.000         425.000      425.000           425.000
mean       0.284           0.676        0.767             0.847
std        0.059           0.057        0.061             0.052
min        0.135           0.518        0.568             0.668
25%        0.245           0.641        0.727             0.811
50%        0.280           0.676        0.771             0.853
75%        0.323           0.712        0.808             0.882
max        0.477           0.825        0.941             0.966


In [3]:
print("=== best plate discipline (lowest chase) ===")
print(prof.nsmallest(10, "chase_pct")[["chase_pct", "zone_swing_pct", "zone_contact_pct", "pitches"]].round(3).to_string())
print()
print("=== highest chase ===")
print(prof.nlargest(10, "chase_pct")[["chase_pct", "zone_swing_pct", "zone_contact_pct", "pitches"]].round(3).to_string())
print()
print("chase vs zone_contact correlation:", prof["chase_pct"].corr(prof["zone_contact_pct"]).round(3))
print("chase vs zone_swing correlation:  ", prof["chase_pct"].corr(prof["zone_swing_pct"]).round(3))

=== best plate discipline (lowest chase) ===
        chase_pct  zone_swing_pct  zone_contact_pct  pitches
batter                                                      
663757      0.135           0.558             0.842      857
621466      0.141           0.662             0.784      766
543257      0.156           0.570             0.850     1032
663457      0.168           0.564             0.892     1665
663656      0.171           0.677             0.887     1362
669003      0.171           0.672             0.763      891
457705      0.171           0.655             0.801     2166
543685      0.174           0.594             0.930      977
664238      0.176           0.650             0.839     1809
624415      0.176           0.554             0.860      956

=== highest chase ===
        chase_pct  zone_swing_pct  zone_contact_pct  pitches
batter                                                      
542194      0.477           0.762             0.807      516
678882      0.460

In [4]:
prof["discipline_gap"] = prof["zone_swing_pct"] - prof["chase_pct"]

print("=== best swing decisions (largest gap) ===")
print(prof.nlargest(10, "discipline_gap")[
    ["discipline_gap", "chase_pct", "zone_swing_pct", "zone_contact_pct", "pitches"]
].round(3).to_string())
print()
print("=== worst swing decisions ===")
print(prof.nsmallest(10, "discipline_gap")[
    ["discipline_gap", "chase_pct", "zone_swing_pct", "zone_contact_pct", "pitches"]
].round(3).to_string())
print()
print(prof["discipline_gap"].describe().round(3))
print()
print("gap vs chase:      ", prof["discipline_gap"].corr(prof["chase_pct"]).round(3))
print("gap vs zone_swing: ", prof["discipline_gap"].corr(prof["zone_swing_pct"]).round(3))
print("gap vs zone_contact:", prof["discipline_gap"].corr(prof["zone_contact_pct"]).round(3))

=== best swing decisions (largest gap) ===
        discipline_gap  chase_pct  zone_swing_pct  zone_contact_pct  pitches
batter                                                                      
621035           0.529      0.192           0.721             0.756     1046
608369           0.527      0.267           0.794             0.884     1914
700250           0.523      0.206           0.729             0.855      725
621466           0.521      0.141           0.662             0.784      766
543760           0.509      0.250           0.759             0.888     2635
668885           0.507      0.181           0.688             0.865      934
663656           0.506      0.171           0.677             0.887     1362
669003           0.501      0.171           0.672             0.763      891
682177           0.501      0.244           0.745             0.803      902
664040           0.499      0.304           0.802             0.788     1632

=== worst swing decisions ===
  

In [5]:
sample = df[df["batter"] == 663757][["batter", "pitcher", "player_name"]].head(3)
print(sample)

       batter  pitcher     player_name
34795  663757   592332  Gausman, Kevin
34796  663757   592332  Gausman, Kevin
34797  663757   592332  Gausman, Kevin


In [6]:
import inspect
import pybaseball

print([n for n in dir(pybaseball) if "lookup" in n.lower() or "id" in n.lower()])

['playerid_lookup', 'playerid_reverse_lookup', 'team_ids', 'teamid_lookup']


In [7]:
import inspect
from pybaseball import playerid_reverse_lookup

print(inspect.signature(playerid_reverse_lookup))
print()
print(playerid_reverse_lookup.__doc__)

(player_ids: List[str], key_type: str = 'mlbam') -> pandas.DataFrame

Retrieve a table of player information given a list of player ids

    :param player_ids: list of player ids
    :type player_ids: list
    :param key_type: name of the key type being looked up (one of "mlbam", "retro", "bbref", or "fangraphs")
    :type key_type: str

    :rtype: :class:`pandas.core.frame.DataFrame`
    


In [8]:
ids = prof.index.tolist()
names = playerid_reverse_lookup(ids, key_type="mlbam")   # 시그니처 보고 조정
print(names.shape)
print(names.head())
print(names.columns.tolist())

Gathering player lookup table. This may take a moment.
(425, 8)
   name_last name_first  key_mlbam key_retro  key_bbref  key_fangraphs  \
0      edman      tommy     669242  edmat001  edmanto01          19470   
1  schneider      davis     676914  schnd001  schneda03          23565   
2      lopez      nicky     670032  lopen001  lopezni01          19339   
3       lowe    brandon     664040  loweb001   lowebr01          18882   
4    goodman     hunter     696100  goodh001  goodmhu01          29715   

   mlb_played_first  mlb_played_last  
0            2019.0           2026.0  
1            2023.0           2026.0  
2            2019.0           2026.0  
3            2018.0           2026.0  
4            2023.0           2026.0  
['name_last', 'name_first', 'key_mlbam', 'key_retro', 'key_bbref', 'key_fangraphs', 'mlb_played_first', 'mlb_played_last']


In [9]:
from src.data.ingestion.statcast_client import project_root

out = project_root() / "data" / "external" / "player_ids.csv"
out.parent.mkdir(parents=True, exist_ok=True)
names.to_csv(out, index=False)
print(out, out.stat().st_size / 1e6, "MB")

/Users/minjong/Projects/mlb-intelligence-lab/data/external/player_ids.csv 0.025478 MB


In [10]:
name_map = names.set_index("key_mlbam")[["name_first", "name_last"]]
prof = prof.join(name_map)
prof["name"] = prof["name_last"].str.title() + ", " + prof["name_first"].str.title()

print(prof.nlargest(10, "discipline_gap")[
    ["name", "discipline_gap", "chase_pct", "zone_swing_pct", "zone_contact_pct"]
].round(3).to_string())

                      name  discipline_gap  chase_pct  zone_swing_pct  zone_contact_pct
batter                                                                                 
621035       Taylor, Chris           0.529      0.192           0.721             0.756
608369       Seager, Corey           0.527      0.267           0.794             0.884
700250           Rice, Ben           0.523      0.206           0.729             0.855
621466         Stewart, Dj           0.521      0.141           0.662             0.784
543760      Semien, Marcus           0.509      0.250           0.759             0.888
668885      Martin, Austin           0.507      0.181           0.688             0.865
663656        Tucker, Kyle           0.506      0.171           0.677             0.887
669003   Mitchell, Garrett           0.501      0.171           0.672             0.763
682177  Schneemann, Daniel           0.501      0.244           0.745             0.803
664040       Lowe, Brandon      